# Final SoH comparison — DataDrivenNN, VerhulstPINN, and DeepHPMNN

This notebook presents the frozen Step 12 comparison. It reads saved reports and figures only: it does **not** load battery data, instantiate checkpoints, select hyperparameters, or retrain models.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
COMPARISON = ROOT / 'artifacts' / 'comparisons' / 'soh'
FIGURES = COMPARISON / 'figures'
summary = json.loads((COMPARISON / 'comparison_summary.json').read_text())
alignment = summary['alignment']

## Cohort and alignment verification

In [ ]:
display(Markdown(f'''
- Alignment key: `{tuple(alignment['key'])}`
- Exact alignment verified: **{alignment['verified']}**
- Identical targets verified: **{alignment['targets_identical']}**
- Test records: **{alignment['records']:,}**
- Test batteries: **{alignment['batteries']}**
- Fixed seeds: **{alignment['seeds']}**
- Cohort fingerprint: `{alignment['cohort_sha256']}`
'''))

## Headline predictive results

Seed mean ± sample SD summarizes three independently trained models. Ensemble metrics are calculated after averaging the three aligned predictions, so the two result types should not be interchanged.

In [ ]:
rows = ['| Model | Seed MAE mean ± SD | Seed RMSE mean ± SD | Seed R² mean ± SD | Ensemble RMSE | Ensemble R² | Ensemble bias |',
        '|---|---:|---:|---:|---:|---:|---:|']
for key in ('data_driven', 'verhulst', 'deephpm'):
    model = summary['models'][key]
    aggregate = model['seed_aggregate_metrics']
    ensemble = model['ensemble_metrics']
    rows.append(
        f"| {model['label']} | {aggregate['mae']['mean']:.6f} ± {aggregate['mae']['sample_std']:.6f} | "
        f"{aggregate['rmse']['mean']:.6f} ± {aggregate['rmse']['sample_std']:.6f} | "
        f"{aggregate['r2']['mean']:.4f} ± {aggregate['r2']['sample_std']:.4f} | "
        f"{ensemble['rmse']:.6f} | {ensemble['r2']:.4f} | {ensemble['bias']:.6f} |"
    )
display(Markdown('\n'.join(rows)))

In [ ]:
display(Image(filename=str(FIGURES / '01_metric_means_with_seed_error_bars.png'), width=1000))

## Prediction agreement and degradation trajectories

All panels use the same held-out cycles. Representative batteries are selected by the lowest, median, and highest mean ensemble RMSE across all three models, rather than by choosing visually favorable examples.

In [ ]:
for name in ('02_measured_vs_predicted_soh', '03_representative_battery_trajectories'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1150))

## Battery-level and distributional behavior

In [ ]:
for name in ('04_per_battery_rmse', '05_error_distributions'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1100))

In [ ]:
rows = ['| Model | Absolute error P50 | P90 | P95 | P99 |', '|---|---:|---:|---:|---:|']
for key in ('data_driven', 'verhulst', 'deephpm'):
    model = summary['models'][key]
    metric = model['ensemble_metrics']
    rows.append(f"| {model['label']} | {metric['absolute_error_p50']:.6f} | {metric['absolute_error_p90']:.6f} | {metric['absolute_error_p95']:.6f} | {metric['absolute_error_p99']:.6f} |")
display(Markdown('### Ensemble absolute-error percentiles\n\n' + '\n'.join(rows)))

## Seed stability and physics consistency

Physics residuals are shown only for VerhulstPINN and DeepHPMNN. A lower residual means closer agreement with that model's governing equation; it does not by itself establish better SoH prediction.

In [ ]:
for name in ('06_seed_stability', '07_physics_residuals'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1100))

## Evidence-based interpretation

On this frozen SoH protocol, VerhulstPINN has the lowest seed-mean and ensemble prediction errors. DeepHPMNN has a substantially smaller physics residual, but its SoH metrics are effectively the same as the DataDrivenNN baseline. This means the current evidence supports a predictive benefit for the selected Verhulst constraint, while the learned DeepHPM dynamics improve equation consistency without a measurable predictive gain in these runs. With only three seeds, these values should be presented as experimental results rather than a formal statistical-significance claim.

## Presentation guidance

Use the metric means figure and measured-versus-predicted figure on the main comparison slide. Use representative trajectories and per-battery RMSE for behavior across cells, and place the physics residual figure on a methodology/diagnostics slide. Vector SVG and PDF versions are available for lossless placement.